In [2]:
import numpy as np
from collections import defaultdict

# ==========================================================
# STEP 1 : INITIAL VOCABULARY
# ==========================================================

vocab = {
    "l o w </w>": 5,
    "l o w e r </w>": 2,
    "n e w e s t </w>": 6,
    "w i d e s t </w>": 3
}

print("========== INITIAL VOCABULARY ==========")
for word, freq in vocab.items():
    print(word, ":", freq)

# ==========================================================
# STEP 2 : WORD FREQUENCIES
# ==========================================================

freq = np.array(list(vocab.values()))

print("\nWord Frequencies :", freq)
print("Total Words :", np.sum(freq))
print("Maximum Frequency :", np.max(freq))

# ==========================================================
# STEP 3 : COUNT PAIRS
# ==========================================================

def get_pair_counts(vocab):

    pairs = defaultdict(int)

    for word, frequency in vocab.items():

        symbols = word.split()

        for i in range(len(symbols)-1):

            pair = (symbols[i], symbols[i+1])

            pairs[pair] += frequency

    return pairs

# ==========================================================
# STEP 4 : MERGE PAIRS
# ==========================================================

def merge_pair(pair, vocab):

    new_vocab = {}

    bigram = " ".join(pair)

    replacement = "".join(pair)

    for word in vocab:

        new_word = word.replace(bigram, replacement)

        new_vocab[new_word] = vocab[word]

    return new_vocab

# ==========================================================
# STEP 5 : PERFORM MERGES
# ==========================================================

merge_rules = []

num_merges = 10

for i in range(num_merges):

    pairs = get_pair_counts(vocab)

    if len(pairs) == 0:
        break

    best_pair = max(pairs, key=pairs.get)

    merge_rules.append(best_pair)

    print("\nMerge", i+1)
    print("Best Pair :", best_pair)
    print("Frequency :", pairs[best_pair])

    vocab = merge_pair(best_pair, vocab)

# ==========================================================
# STEP 6 : FINAL VOCABULARY
# ==========================================================

final_vocab = set()

for word in vocab:

    final_vocab.update(word.split())

print("\n========== FINAL VOCABULARY ==========")
print(sorted(final_vocab))
print("Vocabulary Size :", len(final_vocab))

# ==========================================================
# STEP 7 : ENCODER
# ==========================================================

def encode(word, merge_rules):

    tokens = list(word)

    tokens.append("</w>")

    for pair in merge_rules:

        i = 0

        while i < len(tokens)-1:

            if tokens[i] == pair[0] and tokens[i+1] == pair[1]:

                tokens = tokens[:i] + ["".join(pair)] + tokens[i+2:]

            else:

                i += 1

    return tokens

# ==========================================================
# STEP 8 : DECODER
# ==========================================================

def decode(tokens):

    word = "".join(tokens)

    return word.replace("</w>","")

# ==========================================================
# STEP 9 : TEST
# ==========================================================

test_word = "newest"

encoded = encode(test_word, merge_rules)

decoded = decode(encoded)

print("\nMerge Rules")

for i, rule in enumerate(merge_rules):

    print(i+1, ":", rule)

print("\nEncoded :", encoded)

print("Decoded :", decoded)

# ==========================================================
# STEP 10 : TOKEN IDs
# ==========================================================

token_vocab = sorted(final_vocab)

token_to_id = {}

id_to_token = {}

for i, token in enumerate(token_vocab):

    token_to_id[token] = i

    id_to_token[i] = token

print("\n========== TOKEN IDs ==========")

for token in token_vocab:

    print(token, "->", token_to_id[token])

# ==========================================================
# STEP 11 : CORPUS TOKEN IDS
# ==========================================================

corpus = []

for word in vocab:

    corpus.extend(word.split())

token_ids = []

for token in corpus:

    token_ids.append(token_to_id[token])

print("\nCorpus")

print(corpus)

print("\nToken IDs")

print(token_ids)

# ==========================================================
# STEP 12 : INPUT TARGET PAIRS
# ==========================================================

X = np.array(token_ids[:-1])

Y = np.array(token_ids[1:])

print("\nInput")

print(X)

print("\nTarget")

print(Y)

# ==========================================================
# STEP 13 : EMBEDDINGS
# ==========================================================

np.random.seed(42)

embedding_dim = 8

vocab_size = len(token_vocab)

embeddings = np.random.randn(vocab_size, embedding_dim)

embedded = embeddings[X]

print("\nEmbedding Shape")

print(embedded.shape)

# ==========================================================
# STEP 14 : POSITIONAL ENCODING
# ==========================================================

sequence_length = len(X)

position = np.arange(sequence_length).reshape(-1,1)

dimension = np.arange(embedding_dim).reshape(1,-1)

angles = position / np.power(10000,(2*(dimension//2))/embedding_dim)

positional = np.where(dimension%2==0,np.sin(angles),np.cos(angles))

embedded = embedded + positional

print("\nPositional Encoding Added")

# ==========================================================
# STEP 15 : CAUSAL MASK
# ==========================================================

mask = np.triu(np.ones((sequence_length,sequence_length)),k=1)

mask[mask==1] = -1e9

print("\nCausal Mask Shape")

print(mask.shape)

# ==========================================================
# STEP 16 : SELF ATTENTION
# ==========================================================

d_model = embedding_dim

WQ = np.random.randn(d_model,d_model)

WK = np.random.randn(d_model,d_model)

WV = np.random.randn(d_model,d_model)

Q = embedded @ WQ

K = embedded @ WK

V = embedded @ WV

scores = (Q @ K.T)/np.sqrt(d_model)

scores = scores + mask

exp_scores = np.exp(scores-np.max(scores,axis=1,keepdims=True))

attention = exp_scores / np.sum(exp_scores,axis=1,keepdims=True)

context = attention @ V

print("\nAttention Shape")

print(attention.shape)

# ==========================================================
# STEP 17 : LINEAR LAYER
# ==========================================================

WO = np.random.randn(d_model,vocab_size)

logits = context @ WO

print("\nLogits Shape")

print(logits.shape)

# ==========================================================
# STEP 18 : SOFTMAX
# ==========================================================

exp_logits = np.exp(logits-np.max(logits,axis=1,keepdims=True))

probabilities = exp_logits / np.sum(exp_logits,axis=1,keepdims=True)

print("\nSoftmax Shape")

print(probabilities.shape)

# ==========================================================
# STEP 19 : CROSS ENTROPY LOSS
# ==========================================================

loss = -np.mean(np.log(probabilities[np.arange(len(Y)),Y]+1e-9))

print("\nCross Entropy Loss")

print(loss)

# ==========================================================
# STEP 20 : NEXT TOKEN PREDICTION
# ==========================================================

predicted_ids = np.argmax(probabilities,axis=1)

predicted_tokens = []

for pid in predicted_ids:

    predicted_tokens.append(id_to_token[pid])

print("\nPredicted Token IDs")

print(predicted_ids)

print("\nPredicted Tokens")

print(predicted_tokens)

print("\nNext Predicted Token")

print(predicted_tokens[-1])

# ==========================================================
# STEP 21 : BACKPROPAGATION
# ==========================================================

print("\nBackpropagation")

print("Gradient calculation and weight updates happen here during training.")

print("\n========== PROGRAM COMPLETED ==========")

========== INITIAL VOCABULARY ==========
l o w </w> : 5
l o w e r </w> : 2
n e w e s t </w> : 6
w i d e s t </w> : 3

Word Frequencies : [5 2 6 3]
Total Words : 16
Maximum Frequency : 6

Merge 1
Best Pair : ('e', 's')
Frequency : 9

Merge 2
Best Pair : ('es', 't')
Frequency : 9

Merge 3
Best Pair : ('est', '</w>')
Frequency : 9

Merge 4
Best Pair : ('l', 'o')
Frequency : 7

Merge 5
Best Pair : ('lo', 'w')
Frequency : 7

Merge 6
Best Pair : ('n', 'e')
Frequency : 6

Merge 7
Best Pair : ('ne', 'w')
Frequency : 6

Merge 8
Best Pair : ('new', 'est</w>')
Frequency : 6

Merge 9
Best Pair : ('low', '</w>')
Frequency : 5

Merge 10
Best Pair : ('w', 'i')
Frequency : 3

========== FINAL VOCABULARY ==========
['</w>', 'd', 'e', 'est</w>', 'low', 'low</w>', 'newest</w>', 'r', 'wi']
Vocabulary Size : 9

Merge Rules
1 : ('e', 's')
2 : ('es', 't')
3 : ('est', '</w>')
4 : ('l', 'o')
5 : ('lo', 'w')
6 : ('n', 'e')
7 : ('ne', 'w')
8 : ('new', 'est</w>')
9 : ('low', '</w>')
10 : ('w', 'i')

Encoded : ['n